# 04. RFM Analysis & Customer Segmentation

## Objective
Compute **Recency, Frequency, and Monetary (RFM)** values for each customer, assign **1–5 quantile scores**, and segment customers into 9 distinct business categories.

### Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option('display.float_format', lambda x: '%.2f' % x)
print("Libraries loaded.")

### Step 2: Load Cleaned Transaction Dataset

In [ ]:
df = pd.read_csv('../data/processed/cleaned_transactions.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f"Loaded {len(df):,} transactions for {df['CustomerID'].nunique():,} unique customers.")

### Step 3: Determine Dynamic Snapshot Reference Date
Reference snapshot date = `max(InvoiceDate) + 1 day`.

In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Dynamic Snapshot Reference Date: {snapshot_date}")

### Step 4: Calculate Customer-Level Recency
Recency = Number of days elapsed between customer's most recent purchase and snapshot date.

In [ ]:
recency_df = df.groupby('CustomerID')['InvoiceDate'].max().reset_index()
recency_df['Recency'] = (snapshot_date - recency_df['InvoiceDate']).dt.days
recency_df.head()

### Step 5: Calculate Customer-Level Frequency
Frequency = Count of unique `InvoiceNo` orders per customer.

In [ ]:
freq_df = df.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
freq_df.columns = ['CustomerID', 'Frequency']
freq_df.head()

### Step 6: Calculate Customer-Level Monetary Value
Monetary = Cumulative sum of `Revenue` generated by customer.

In [ ]:
monetary_df = df.groupby('CustomerID')['Revenue'].sum().reset_index()
monetary_df.columns = ['CustomerID', 'Monetary']
monetary_df['Monetary'] = monetary_df['Monetary'].round(2)
monetary_df.head()

### Step 7: Combine into Unified RFM DataFrame
Merge Recency, Frequency, and Monetary into a single DataFrame.

In [ ]:
rfm = recency_df[['CustomerID', 'Recency']].merge(freq_df, on='CustomerID').merge(monetary_df, on='CustomerID')
rfm.head()

### Step 8: Inspect Summary Statistics of Raw RFM Metrics

In [ ]:
rfm[['Recency', 'Frequency', 'Monetary']].describe()

### Step 9: Calculate R_Score (1 to 5)
Recency is reversed: Lower recency days = more recent purchase = Score 5.

In [ ]:
r_rank = rfm['Recency'].rank(method='first', ascending=True)
rfm['R_Score'] = pd.qcut(r_rank, q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm[['Recency', 'R_Score']].head()

### Step 10: Calculate F_Score (1 to 5)
Higher order frequency = Score 5.

In [ ]:
f_rank = rfm['Frequency'].rank(method='first', ascending=True)
rfm['F_Score'] = pd.qcut(f_rank, q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm[['Frequency', 'F_Score']].head()

### Step 11: Calculate M_Score (1 to 5)
Higher total spending = Score 5.

In [ ]:
m_rank = rfm['Monetary'].rank(method='first', ascending=True)
rfm['M_Score'] = pd.qcut(m_rank, q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm[['Monetary', 'M_Score']].head()

### Step 12: Combine Scores into RFM_Score String & RFM_Total_Score

In [ ]:
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm['RFM_Total_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']
rfm.head()

### Step 13: Define Segmentation Rules Function
Map RFM score combinations to transparent business segments.

In [ ]:
def assign_segment(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    if r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customers'
    if m >= 4:
        return 'Big Spenders'
    if r >= 4 and f in [2, 3]:
        return 'Potential Loyalists'
    if r >= 4 and f == 1:
        return 'New Customers'
    if r in [2, 3] and f in [2, 3]:
        return 'Need Attention'
    if r <= 2 and (f >= 3 or m >= 3):
        return 'At Risk'
    if r == 1 and f <= 2 and m <= 2:
        return 'Lost Customers'
    return 'Hibernating / Low Value'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)
rfm['Segment'].value_counts()

### Step 14: Analyze Segment Metrics & Revenue Contribution
Compute customer counts, revenue total, revenue %, and average R, F, M per segment.

In [ ]:
total_revenue = rfm['Monetary'].sum()
total_custs = len(rfm)

seg_summary = rfm.groupby('Segment').agg(
    Customer_Count=('CustomerID', 'count'),
    Total_Revenue=('Monetary', 'sum'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean')
).reset_index()

seg_summary['Customer_%'] = (seg_summary['Customer_Count'] / total_custs * 100).round(2)
seg_summary['Revenue_%'] = (seg_summary['Total_Revenue'] / total_revenue * 100).round(2)

seg_summary = seg_summary.sort_values(by='Total_Revenue', ascending=False).reset_index(drop=True)
seg_summary

### Step 15: Visualization — Customer Count & Revenue Contribution by Segment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(data=seg_summary, x='Customer_Count', y='Segment', hue='Segment', palette='viridis', ax=axes[0], legend=False)
axes[0].set_title('Customer Count by Segment', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Customers')

sns.barplot(data=seg_summary, x='Total_Revenue', y='Segment', hue='Segment', palette='magma', ax=axes[1], legend=False)
axes[1].set_title('Total Revenue ($) by Segment', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Total Revenue ($)')

plt.tight_layout()
plt.show()

### Step 16: Save RFM Customer Data
Export to `../data/processed/rfm_customers.csv`.

In [ ]:
rfm.to_csv('../data/processed/rfm_customers.csv', index=False)
print("Saved rfm_customers.csv successfully.")